# 02 - Testing models on job ads they have not seen

This notebook provides the same fair test for every model. It runs the two logistic-regression reference models; later model notebooks can use the same helpers and saved folds.

**What happens in each test round**
1. Put some training ads aside for scoring. Related ads always stay together.
2. Using only the remaining ads, try a few model settings. Each setting predicts ads it did not train on.
3. Keep the setting with the best average PR-AUC, then use its unseen-ad predictions to choose the flagging cutoff.
4. Train that model on all the available ads for this round. Score the ads put aside in step 1, using the cutoff already chosen.

The ads in step 4 have no say in choosing settings or the cutoff. The separate reserved test file is not opened during development.

## 1. Features carried forward

Notebook 01 built the code in `src/features.py`:
- TF-IDF represents the words and short phrases in an ad. TF-IDF-only logistic regression is the reference to beat (about 0.58 mean PR-AUC in the earlier repeated tests).
- A2 adds the original fields, missing-field/contact/salary features, and writing-style features. It is the configured primary set, but has not demonstrated an improvement over TF-IDF alone.
- A3 also adds a learned phrase score; earlier tests did not establish a benefit from it.
- Text vocabularies, category encodings, scaling and any learned phrase scores are rebuilt inside **every** training split, including the smaller splits used to choose settings. The full unfitted pipeline must be passed to the evaluation helper.

There are only about 331 training groups containing fraud. Results can vary substantially between rounds; small differences need uncertainty intervals and careful interpretation.

## 2. Load training data only

Run notebook 00 first. The reserved `d1_test.csv` is read only in the final, switched-off section below.

In [ ]:
import json
import re
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.base import clone
from sklearn.metrics import average_precision_score, precision_recall_curve
from IPython.display import display

ROOT = next((p for p in [Path.cwd(), *Path.cwd().parents]
             if (p / 'data' / 'processed' / 'd1_train.csv').exists()), None)
if ROOT is None:
    raise FileNotFoundError('Run notebook 00 first to create data/processed/d1_train.csv.')
sys.path.insert(0, str(ROOT))

from src.features import (FEATURE_SETS, GROUP, PRIMARY_SET, SEED, TARGET, add_country,
                          build_inputs, logistic_pipeline, make_features)
from src.evaluation import (best_f1_threshold, cross_validate as evaluate_folds,
                            fold_signature, make_folds, paired_comparison,
                            pooled_metrics as pooled_score_summary, score_summary,
                            select_settings, summarise)

sns.set_theme(style='whitegrid')
pd.set_option('display.max_columns', 60)
pd.set_option('display.width', 170)

PROCESSED_DIR = ROOT / 'data' / 'processed'
FIG_DIR = ROOT / 'reports' / 'figures'
TAB_DIR = ROOT / 'reports' / 'tables'
for folder in (FIG_DIR, TAB_DIR):
    folder.mkdir(parents=True, exist_ok=True)

train = add_country(pd.read_csv(PROCESSED_DIR / 'd1_train.csv'))
X_train = build_inputs(train)
y_train = train[TARGET].to_numpy()
groups_train = train[GROUP].to_numpy()
assert train['job_id'].is_unique, 'Training job IDs must be unique.'
print('Project folder:', ROOT)
print(f'Training: {len(train):,} ads | fraud {y_train.mean():.2%} | groups {train[GROUP].nunique():,}')
print('The reserved test file has not been opened.')

## 3. Use the same test rounds for every model

The training data is divided into five parts. We score each part once, keeping it out of training while it is scored. We repeat this with five different arrangements to see how much the results vary.

Within each round, three smaller grouped splits choose settings and the cutoff. These are the **inner** splits; the scored round is the **outer** split. Every row is predicted once per repeat, and related ads never appear on both sides of any split.

All models use `FOLDS`. The exact row assignments are saved in the ignored processed-data folder. The code rejects missing predictions, overlapping groups, or a split without both classes.

`MODEL_PARAM_GRID` tries three values for logistic regression's `C` setting. Other model factories can supply their own small grid, using names such as `model__max_depth`. Ranking settings by inner PR-AUC keeps the outer answers out of this choice.

The full run requires many more fits than notebook 01. For a development check, use one repeat and a one-value grid; return to five repeats for reported comparisons. Fix these choices before comparing models.

In [ ]:
N_SPLITS = 5
N_REPEATS = 5
INNER_SPLITS = 3
MODEL_PARAM_GRID = {'model__C': [0.1, 0.3, 1.0]}

FOLDS = make_folds(groups_train, y_train, n_splits=N_SPLITS, n_repeats=N_REPEATS, seed=SEED)
fold_table = pd.DataFrame([
    {'repeat': r, 'fold': k, 'train_rows': len(f), 'val_rows': len(v),
     'val_fraud': int(y_train[v].sum()), 'val_fraud_rate': y_train[v].mean(),
     'val_groups': len(np.unique(groups_train[v])),
     'fold_signature': fold_signature(y_train, groups_train, f, v)}
    for r, k, f, v in FOLDS
])
fold_table.to_csv(TAB_DIR / 'model_fold_plan.csv', index=False)
assignments = pd.concat([
    pd.DataFrame({'repeat': r, 'fold': k, 'job_id': train.iloc[v]['job_id'].to_numpy(),
                  'group_id': groups_train[v]}) for r, k, _, v in FOLDS
], ignore_index=True)
assignments.to_csv(PROCESSED_DIR / 'model_cv_assignments.csv', index=False)
print(f'{len(FOLDS)} scoring rounds ({N_REPEATS} repeats x {N_SPLITS} parts); {INNER_SPLITS} inner splits per round')
display(fold_table.groupby('repeat')[['val_rows', 'val_fraud', 'val_fraud_rate']].agg(['min', 'max']).round(3))

## 4. Measure performance without reusing the answers

We report:
- **PR-AUC:** how well the model puts fake ads ahead of genuine ads across cutoffs; higher is better.
- **Precision:** of the ads flagged, how many are fake?
- **Recall:** of all the fake ads, how many were caught?
- **F1:** a combined precision/recall score. The cutoff is chosen to maximise F1 using the inner unseen-ad predictions.
- **Review budget:** if reviewers read only the highest-scoring 5% of ads, how many are fake and how much fraud is caught?
- **ROC AUC:** an additional ranking measure.

`score_summary` never chooses a cutoff from the examples it scores. Without an explicit cutoff, it returns only ranking and review-budget measures. Pooled predictions are useful for curves, but cannot choose a cutoff for their own reported precision/recall.

The per-round table records the settings and cutoff chosen before scoring. Paired comparisons check that both models used the same rows and rounds. Their uncertainty interval allows approximately for reuse of training data; it is not an exact guarantee. The spread across rounds is not itself a confidence interval.

In [ ]:
def cross_validate(make_model, X=None, y=None, groups=None, folds=None, **kwargs):
    # Every candidate is a complete, unfitted feature + classifier pipeline.
    kwargs.setdefault('inner_splits', INNER_SPLITS)
    kwargs.setdefault('seed', SEED)
    return evaluate_folds(
        make_model, X_train if X is None else X, y_train if y is None else y,
        groups_train if groups is None else groups, folds=FOLDS if folds is None else folds,
        **kwargs,
    )


def pooled_metrics(oof, y=None):
    return pooled_score_summary(oof, y_train if y is None else y)


def plot_pr_curves(oofs, path=None):
    fig, ax = plt.subplots(figsize=(5.4, 4.2))
    for name, oof in oofs.items():
        precision, recall, _ = precision_recall_curve(y_train, oof[0])
        ax.plot(recall, precision, label=f'{name} (PR-AUC {average_precision_score(y_train, oof[0]):.2f})')
    ax.axhline(y_train.mean(), color='#c0504d', linestyle='--', linewidth=1, label='no skill')
    ax.set(xlabel='Recall (fraud class)', ylabel='Precision (fraud class)',
           title='Predictions on unseen ads: first repeat')
    ax.legend(frameon=False)
    plt.tight_layout()
    if path:
        plt.savefig(path, dpi=200)
    plt.show()

## 5. Check the process with the two reference models

Try TF-IDF-only logistic regression and A2 logistic regression. Each chooses its `C` setting and cutoff independently inside the same outer rounds. No outer validation labels choose either value.

Saved outputs include the per-round scores, candidate-setting diagnostics, a comparison table, and a precision-recall figure. Row-level predictions and split assignments go in `data/processed/`, which is ignored by Git.

Candidate-setting scores explain what was chosen; report the outer scores as performance. Choosing among many models using outer results can still overfit the development process, so keep comparisons limited and reserve the separate test file for the final decision.

In [ ]:
RESULTS = {}


def run_model(name, make_model, param_grid=None, **kwargs):
    fold_scores, oof = cross_validate(make_model, param_grid=param_grid, **kwargs)
    RESULTS[name] = {'folds': fold_scores, 'oof': oof, 'make_model': make_model, 'param_grid': param_grid}
    stem = re.sub(r'[^a-z0-9]+', '_', name.lower()).strip('_')
    saved_scores = fold_scores.copy()
    saved_scores['selected_params'] = saved_scores['selected_params'].map(lambda p: json.dumps(p, sort_keys=True))
    saved_scores.to_csv(TAB_DIR / f'model_{stem}_fold_scores.csv', index=False)
    candidates = fold_scores.attrs['inner_candidates'].copy()
    if not candidates.empty:
        candidates['params'] = candidates['params'].map(lambda p: json.dumps(p, sort_keys=True))
        candidates.to_csv(TAB_DIR / f'model_{stem}_inner_settings.csv', index=False)
    pd.DataFrame([
        {'repeat': r, 'job_id': job, 'probability': p}
        for r, predictions in enumerate(oof)
        for job, p in zip(train['job_id'], predictions)
    ]).to_csv(PROCESSED_DIR / f'model_{stem}_oof.csv', index=False)
    return summarise(fold_scores, name)


baseline = run_model('LogReg, TF-IDF only', lambda: logistic_pipeline(['tfidf']), param_grid=MODEL_PARAM_GRID)
primary = run_model(f'LogReg, {PRIMARY_SET}', lambda: logistic_pipeline(FEATURE_SETS[PRIMARY_SET]), param_grid=MODEL_PARAM_GRID)
display(pd.concat([baseline, primary], axis=1).T.round(3))
print(f'A no-skill ranking scores about the fraud rate: {y_train.mean():.3f}')

In [ ]:
def results_table():
    return pd.DataFrame([summarise(v['folds'], name) for name, v in RESULTS.items()]).sort_values('pr_auc', ascending=False)


def compare_to(reference, names=None):
    names = [n for n in (RESULTS if names is None else names) if n != reference]
    return pd.DataFrame({n: paired_comparison(RESULTS[n]['folds'], RESULTS[reference]['folds']) for n in names}).T


summary = results_table()
comparison = compare_to('LogReg, TF-IDF only')
display(summary.round(3))
display(comparison.round(3))
summary.to_csv(TAB_DIR / 'model_comparison.csv')
comparison.to_csv(TAB_DIR / 'model_paired_vs_tfidf.csv')
plot_pr_curves({k: v['oof'] for k, v in RESULTS.items()}, FIG_DIR / 'model_pr_curves.png')
print('An interval crossing zero does not establish which model is better.')

## 6. Keep the reserved test set for the final check

Leave the switch off during development. Choose `FINAL_MODEL_NAME` only after the model comparison is complete.

When the team is ready for the final test, the chosen model uses the training data to choose its settings and cutoff with grouped unseen-ad predictions. Those choices are written down **before** opening the reserved test file. The model is then trained on all training ads and tested once. Do not adjust it in response to that final result.

In [ ]:
EVALUATE_ON_TEST = False
FINAL_MODEL_NAME = None      # Set to a key in RESULTS only after model selection is complete.


def evaluate_on_test(make_model, params, threshold):
    # Settings and threshold must already be fixed using training data only.
    model = clone(make_model()).set_params(**params).fit(X_train, y_train)
    test = add_country(pd.read_csv(PROCESSED_DIR / 'd1_test.csv'))
    if not set(train[GROUP]).isdisjoint(test[GROUP]) or not set(train['job_id']).isdisjoint(test['job_id']):
        raise ValueError('The reserved test set shares ads or groups with training.')
    p = model.predict_proba(build_inputs(test))[:, 1]
    return pd.Series(score_summary(test[TARGET].to_numpy(), p, threshold=threshold))


if EVALUATE_ON_TEST:
    if FINAL_MODEL_NAME not in RESULTS:
        raise ValueError('Choose FINAL_MODEL_NAME from the completed model comparison first.')
    chosen = RESULTS[FINAL_MODEL_NAME]
    selection = select_settings(chosen['make_model'], X_train, y_train, groups_train,
                                param_grid=chosen['param_grid'], inner_splits=INNER_SPLITS, seed=SEED)
    frozen = {'model': FINAL_MODEL_NAME, 'params': selection.params, 'threshold': selection.threshold,
              'seed': SEED, 'inner_splits': INNER_SPLITS}
    (TAB_DIR / 'final_model_settings.json').write_text(json.dumps(frozen, indent=2), encoding='utf-8')
    final_metrics = evaluate_on_test(chosen['make_model'], selection.params, selection.threshold)
    final_metrics.to_csv(TAB_DIR / 'final_test_metrics.csv')
    display(final_metrics.round(3).to_frame('final test'))
else:
    print('Reserved test evaluation is switched off; the test file has not been opened.')